# Recognition-time (RTF) measurement for the five CG-ROVER-CRF systems — all 100 recordings
Kaggle: **Settings → Accelerator → GPU T4 x1** (one T4, not T4 x2), **Internet → On**, and add your audio dataset (the 100 WAV files) via *Add Input*.
Timing includes everything the original pipeline did per file (decoding the file, VAD or 30-s chunking, inference); model loading and one warm-up call are excluded.
The run is resumable: results are appended to `rtf_per_file.csv` after every file, and files already done are skipped if you re-run a cell. Expected total: about 2-2.5 hours on a T4; use *Save Version → Save & Run All (Commit)* to let it run in the background.
Systems: ksc2-VAD, kk-turbo-VAD, kk-turbo-chunk, ksc2-chunk, MMS-VAD (same code as in the experiment notebooks).

In [1]:
!pip -q install pydub transformers accelerate
import os, re, time, glob, shutil, tempfile, json
import numpy as np, pandas as pd, torch
from pydub import AudioSegment
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (switch on GPU!)')

/usr/local/lib/python3.13/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.13/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


GPU: Tesla T4


## 1. Find the audio and choose the test files

In [2]:
AUDIO_ROOT = '/kaggle/input'
USED = ['10m/10m_001', '10m/10m_002', '10m/10m_003', '10m/10m_004', '10m/10m_005', '10m/10m_006', '10m/10m_007', '10m/10m_008', '10m/10m_009', '10m/10m_010', '1m/1m_001', '1m/1m_002', '1m/1m_003', '1m/1m_004', '1m/1m_005', '1m/1m_006', '1m/1m_007', '1m/1m_008', '1m/1m_009', '1m/1m_010', '1m/1m_011', '1m/1m_012', '1m/1m_013', '1m/1m_014', '1m/1m_015', '1m/1m_016', '1m/1m_017', '1m/1m_018', '1m/1m_019', '1m/1m_020', '1m/1m_021', '1m/1m_022', '1m/1m_023', '1m/1m_024', '1m/1m_025', '2m/2m_001', '2m/2m_002', '2m/2m_003', '2m/2m_004', '2m/2m_005', '2m/2m_006', '2m/2m_007', '2m/2m_008', '2m/2m_009', '2m/2m_010', '2m/2m_011', '2m/2m_012', '2m/2m_013', '2m/2m_014', '2m/2m_015', '2m/2m_016', '2m/2m_017', '2m/2m_018', '2m/2m_019', '2m/2m_020', '30s/30s_001', '30s/30s_002', '30s/30s_003', '30s/30s_004', '30s/30s_005', '30s/30s_006', '30s/30s_007', '30s/30s_008', '30s/30s_009', '30s/30s_010', '30s/30s_011', '30s/30s_012', '30s/30s_013', '30s/30s_014', '30s/30s_015', '30s/30s_016', '30s/30s_017', '30s/30s_018', '30s/30s_019', '30s/30s_020', '30s/30s_021', '30s/30s_022', '30s/30s_023', '30s/30s_024', '30s/30s_025', '30s/30s_026', '30s/30s_027', '30s/30s_028', '30s/30s_029', '30s/30s_030', '5m/5m_001', '5m/5m_002', '5m/5m_003', '5m/5m_004', '5m/5m_005', '5m/5m_006', '5m/5m_007', '5m/5m_008', '5m/5m_009', '5m/5m_010', '5m/5m_011', '5m/5m_012', '5m/5m_013', '5m/5m_014', '5m/5m_015']
wavs = sorted(glob.glob(AUDIO_ROOT + '/**/*.wav', recursive=True))
byid = {}
for p in wavs:
    key = os.path.basename(os.path.dirname(p)) + '/' + os.path.splitext(os.path.basename(p))[0]
    byid.setdefault(key, p)
missing = [u for u in USED if u not in byid]
print('listed:', len(USED), '| found:', len(USED) - len(missing), '| missing:', missing)
info = [(byid[u], len(AudioSegment.from_file(byid[u])) / 1000.0) for u in USED if u in byid]
sel = info                      # all 100 recordings, in the listed order
print('files: %d | total audio: %.1f min' % (len(sel), sum(d for _, d in sel) / 60))

listed: 100 | found: 100 | missing: []
files: 100 | total audio: 259.2 min


## 2. Shared code (copied from the experiment notebooks)

In [3]:
def vad_segment(x, sr=16000, win=0.025, min_sil=0.30, max_len=18.0, min_len=1.0, pad=0.25):
    n = int(win * sr)
    if len(x) < n * 2:
        return [(0, len(x))]
    frames = x[: len(x) // n * n].reshape(-1, n)
    e = np.sqrt((frames ** 2).mean(axis=1) + 1e-12)
    floor = np.percentile(e, 10); speech = np.percentile(e, 75)
    thr = floor + 0.12 * max(speech - floor, 1e-6)
    sil = e < thr
    need = max(1, int(min_sil / win))
    cuts, run = [], 0
    for i, s in enumerate(sil):
        if s: run += 1
        else:
            if run >= need: cuts.append((i - run // 2) * n)
            run = 0
    bounds = sorted({0, len(x)} | {c for c in cuts if 0 < c < len(x)})
    segs, a = [], bounds[0]
    for b in bounds[1:]:
        if (b - a) / sr < min_len and b != bounds[-1]:
            continue
        while (b - a) / sr > max_len:
            segs.append((a, a + int(max_len * sr))); a += int(max_len * sr)
        segs.append((a, b)); a = b
    return [(max(0, s - int(pad * sr)), min(len(x), e2 + int(pad * sr))) for s, e2 in segs]

def load_audio(path, sr=16000):
    a = AudioSegment.from_file(path).set_channels(1).set_frame_rate(sr)
    x = np.array(a.get_array_of_samples()).astype(np.float32)
    return x / float(1 << (8 * a.sample_width - 1)), sr

CHUNK_MS = 30_000
GEN = dict(language="kk", task="transcribe")

def make_whisper_fns(asr):
    def chunk_fn(audio_path):
        audio = AudioSegment.from_file(audio_path)
        tmp = tempfile.mkdtemp(prefix="chunks_"); texts = []
        try:
            for start in range(0, max(len(audio), 1), CHUNK_MS):
                p = os.path.join(tmp, f"c_{start:08d}.wav")
                audio[start:start + CHUNK_MS].export(p, format="wav")
                texts.append(str(asr(p, generate_kwargs=GEN)["text"]).strip())
        finally:
            shutil.rmtree(tmp, ignore_errors=True)
        return " ".join(t for t in texts if t)
    def vad_fn(audio_path):
        x, sr = load_audio(audio_path); texts = []
        for s, e in vad_segment(x, sr):
            c = x[s:e]
            if len(c) < sr * 0.2: continue
            texts.append(str(asr({"array": c, "sampling_rate": sr}, generate_kwargs=GEN)["text"]).strip())
        return " ".join(t for t in texts if t)
    return chunk_fn, vad_fn

CSV = 'rtf_per_file.csv'
rows = pd.read_csv(CSV).to_dict('records') if os.path.exists(CSV) else []
def timed(system, fn):
    done = {r['file'] for r in rows if r['system'] == system}
    todo = [(p, d) for p, d in sel if os.path.basename(os.path.dirname(p)) + '/' + os.path.basename(p) not in done]
    print(system, ': done', len(done), '| to do', len(todo))
    if not todo: return
    fn(min(sel, key=lambda t: t[1])[0]); torch.cuda.synchronize()    # warm-up (not counted)
    for p, dur in todo:
        torch.cuda.synchronize(); t0 = time.perf_counter()
        txt = fn(p)
        torch.cuda.synchronize(); dt = time.perf_counter() - t0
        rows.append(dict(system=system, file=os.path.basename(os.path.dirname(p)) + '/' + os.path.basename(p), audio_s=dur, time_s=dt, words=len(txt.split())))
        if len(rows) % 10 == 0: pd.DataFrame(rows).to_csv(CSV, index=False)
    print(f"{system}: {len(sel)} files, RTF {sum(r['time_s'] for r in rows if r['system']==system)/sum(r['audio_s'] for r in rows if r['system']==system):.3f}")
    pd.DataFrame(rows).to_csv('rtf_per_file.csv', index=False)

## 3. Whisper systems: ksc2 (chunk + VAD)

In [4]:
from transformers import pipeline
asr = pipeline("automatic-speech-recognition", model="abilmansplus/whisper-turbo-ksc2",
               torch_dtype=torch.float16, device=0)
chunk_fn, vad_fn = make_whisper_fns(asr)
timed('ksc2-VAD', vad_fn)
timed('ksc2-chunk', chunk_fn)
del asr; torch.cuda.empty_cache()

config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.24GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

...

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

ksc2-VAD : done 0 | to do 100


[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


ksc2-VAD: 100 files, RTF 0.133
ksc2-chunk : done 0 | to do 100
ksc2-chunk: 100 files, RTF 0.079


## 4. Whisper systems: kk-turbo (chunk + VAD)

In [5]:
asr = pipeline("automatic-speech-recognition", model="shyngys879/kazakh-whisper-large-v3-turbo",
               torch_dtype=torch.float16, device=0)
chunk_fn, vad_fn = make_whisper_fns(asr)
timed('kk-turbo-VAD', vad_fn)
timed('kk-turbo-chunk', chunk_fn)
del asr; torch.cuda.empty_cache()

config.json:   0%|          | 0.00/1.22k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.62GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/587 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/2.88k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.93M [00:00<?, ?B/s]

processor_config.json:   0%|          | 0.00/410 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

kk-turbo-VAD : done 0 | to do 100
kk-turbo-VAD: 100 files, RTF 0.140
kk-turbo-chunk : done 0 | to do 100
kk-turbo-chunk: 100 files, RTF 0.094


## 5. MMS-1b-all (VAD)

In [6]:
from transformers import Wav2Vec2ForCTC, AutoProcessor
processor = AutoProcessor.from_pretrained("facebook/mms-1b-all", target_lang="kaz")
mms = Wav2Vec2ForCTC.from_pretrained("facebook/mms-1b-all", target_lang="kaz", ignore_mismatched_sizes=True).to("cuda").eval().half()

@torch.no_grad()
def mms_arr(chunk, sr=16000):
    iv = processor(chunk, sampling_rate=sr, return_tensors="pt")["input_values"].to("cuda").half()
    ids = torch.argmax(mms(iv).logits, dim=-1)[0]
    return processor.decode(ids).strip()

def mms_vad(audio_path):
    x, sr = load_audio(audio_path); texts = []
    for s, e in vad_segment(x, sr):
        c = x[s:e]
        if len(c) < sr * 0.2: continue
        texts.append(mms_arr(c, sr))
    return " ".join(t for t in texts if t)

timed('MMS-VAD', mms_vad)

preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.04k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.34M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

adapter.kaz.safetensors: reconstructing file:   0%|          |  0.00B / 9.19MB            

adapter.kaz.safetensors: downloading bytes:           |  0.00B            

MMS-VAD : done 0 | to do 100
MMS-VAD: 100 files, RTF 0.019


## 6. Summary

In [7]:
df = pd.DataFrame(rows)
g = df.groupby('system').agg(files=('file','count'), audio_s=('audio_s','sum'), time_s=('time_s','sum'))
g['RTF'] = g.time_s / g.audio_s
g['max_file_s'] = df.groupby('system').time_s.max()
g['audio_min'] = g.audio_s / 60; g['time_min'] = g.time_s / 60
g = g.loc[['ksc2-VAD','kk-turbo-VAD','kk-turbo-chunk','ksc2-chunk','MMS-VAD']]
display(g.round(3))
rtf_sum = g.RTF.sum()
print('Sum of RTF over the five systems (sequential on one T4): %.3f' % rtf_sum)
print('For 259 min of audio: %.1f min of recognition' % (259 * rtf_sum))
print('Combination step RTF = 0.005  ->  share of total cost: %.1f%%' % (100 * 0.005 / (rtf_sum + 0.005)))
print('GPU:', torch.cuda.get_device_name(0))
g.round(4).to_csv('rtf_summary.csv')
print(open('rtf_summary.csv').read())

,files,audio_s,time_s,RTF,max_file_s,audio_min,time_min
system,,,,,,,
ksc2-VAD,100,15549.23,2073.893,0.133,96.331,259.154,34.565
kk-turbo-VAD,100,15549.23,2178.472,0.140,96.378,259.154,36.308
kk-turbo-chunk,100,15549.23,1463.224,0.094,68.400,259.154,24.387
ksc2-chunk,100,15549.23,1224.178,0.079,50.221,259.154,20.403
MMS-VAD,100,15549.23,300.487,0.019,12.252,259.154,5.008


Sum of RTF over the five systems (sequential on one T4): 0.466
For 259 min of audio: 120.6 min of recognition
Combination step RTF = 0.005  ->  share of total cost: 1.1%
GPU: Tesla T4
system,files,audio_s,time_s,RTF,max_file_s,audio_min,time_min
ksc2-VAD,100,15549.23,2073.8934,0.1334,96.3312,259.1538,34.5649
kk-turbo-VAD,100,15549.23,2178.4722,0.1401,96.3783,259.1538,36.3079
kk-turbo-chunk,100,15549.23,1463.2236,0.0941,68.3997,259.1538,24.3871
ksc2-chunk,100,15549.23,1224.1776,0.0787,50.2207,259.1538,20.403
MMS-VAD,100,15549.23,300.4867,0.0193,12.2519,259.1538,5.0081



Download `rtf_per_file.csv` and `rtf_summary.csv` from the *Output* tab (or just paste the summary table back).